## Module 3: Outcome Embedding Model


Based on the following concept: outcome_embedding = encoder(future_trajectory)

Based on latest research:
- Contrastive learning (SimCLR, MoCo principles)
- Transformer encoders (attention over trajectories)
- Self-supervised pre-training (BERT-style for healthcare)

Latest Research Trends:

1. Self-Supervised Learning in Healthcare (2024-2025)
- Pre-training on unlabeled data
- Contrastive learning for medical time series
- Our Module 3 aligns with this trend
    - Architecture inspired by Vision Transformer / BERT
    - Processes FUTURE trajectory segments (next 24-48h)
    - Two heads: mortality (anchor) + outcome embedding
    - Contrastive learning separates good/bad outcomes

3. Multi-Objective RL (2024)
- NO scalarization (let policy decide trade-offs)
- Vector-valued rewards
- Our Module 5 uses this

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from tqdm import tqdm
from typing import List, Tuple, Dict, Optional
from dataclasses import dataclass, field
from copy import deepcopy
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors, KernelDensity
from collections import defaultdict, Counter

import warnings
warnings.filterwarnings('ignore')

import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cpu


In [2]:
# ====================== GLOBAL DATA CONFIG ======================
DATA_DIR = '../../../../../data/'
print(f"✅ Using DATA_DIR: {DATA_DIR}")

if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(f"❌ Data folder not found at {DATA_DIR}\n"
                            f"Please verify the path above.")

# Quick sanity check of required files
required_files = ['ADMISSIONS.csv', 'PATIENTS.csv', 'ICUSTAYS.csv',
                  'DIAGNOSES_ICD.csv', 'CHARTEVENTS.csv', 'LABEVENTS.csv',
                  'OUTPUTEVENTS.csv', 'INPUTEVENTS_MV.csv']
missing = [f for f in required_files if not os.path.exists(os.path.join(DATA_DIR, f))]
if missing:
    print(f"⚠️  Missing files: {missing}")
else:
    print("✅ All required CSV files found.")

✅ Using DATA_DIR: ../../../../../data/
✅ All required CSV files found.


In [3]:
# ====================== IMPORT SHARED UTILITIES ======================
# Import shared classes from utils.data_pipeline module
import os
import sys

notebook_dir = os.getcwd()
src_dir = os.path.abspath(os.path.join(notebook_dir, '..'))

if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from utils.data_pipeline import (
    CohortExtractor, FeatureExtractor, FeatureConfig, FeatureImputer,
    fit_imputer_on_cohort, ActionBins, ActionDiscretizer, Trajectory,
    TrajectoryBuilder, DatasetSplitter, FeatureNormalizer
)
print("✅ Successfully imported shared utilities from utils.data_pipeline")


✅ Successfully imported shared utilities from utils.data_pipeline


In [4]:
class OutcomeEmbeddingModel(nn.Module):
    """
    Module 3: Embeds future patient trajectories into a clinically meaningful outcome space.
    Supports saving and loading of trained weights + learned good_outcome_reference.
    """

    def __init__(self,
                 d_state: int = 76,
                 d_embed: int = 64,
                 d_hidden: int = 256,
                 n_layers: int = 3,
                 n_heads: int = 8,
                 dropout: float = 0.1):
        super().__init__()
        
        self.d_state = d_state
        self.d_embed = d_embed
        self.d_hidden = d_hidden
        
        # State embedding
        self.state_embed = nn.Sequential(
            nn.Linear(d_state, d_hidden),
            nn.LayerNorm(d_hidden),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        # Transformer encoder for future trajectory
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_hidden,
            nhead=n_heads,
            dim_feedforward=4 * d_hidden,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        self.trajectory_encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        
        # Outcome projection
        self.outcome_projection = nn.Sequential(
            nn.Linear(d_hidden, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, d_embed)
        )
        
        # Mortality prediction head
        self.mortality_head = nn.Sequential(
            nn.Linear(d_hidden, d_hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden // 2, 1)
        )
        
        # Learnable reference vector for "good outcome"
        self.register_buffer('good_outcome_reference', torch.zeros(d_embed))
        
        print(f"✅ OutcomeEmbeddingModel initialized (d_state={d_state})")
        print(f"   Hidden: {d_hidden} → Embed: {d_embed}")
        print(f"   Transformer: {n_layers} layers, {n_heads} heads")

    def forward(self, future_states: torch.Tensor, mask: Optional[torch.Tensor] = None):
        B, T, D = future_states.shape
        assert D == self.d_state, f"Expected d_state={self.d_state}, got {D}"
        
        x = self.state_embed(future_states)
        
        if mask is not None:
            attn_mask = ~mask.bool()
        else:
            attn_mask = None
            
        h = self.trajectory_encoder(x, src_key_padding_mask=attn_mask)
        
        # Masked mean pooling
        if mask is not None:
            h_pooled = (h * mask.unsqueeze(-1)).sum(1) / (mask.sum(1, keepdim=True) + 1e-8)
        else:
            h_pooled = h.mean(1)
        
        outcome_embed = self.outcome_projection(h_pooled)
        outcome_embed = F.normalize(outcome_embed, dim=-1)
        
        mortality_logit = self.mortality_head(h_pooled)
        
        return outcome_embed, mortality_logit

    def compute_outcome_similarity(self, outcome_embedding: torch.Tensor) -> torch.Tensor:
        """Cosine similarity to learned good outcome reference"""
        return F.cosine_similarity(
            outcome_embedding,
            self.good_outcome_reference.unsqueeze(0),
            dim=-1
        )

    # ====================== SAVE / LOAD ======================
    
    def save(self, path: str):
        """Save model weights and good outcome reference"""
        torch.save({
            'model_state_dict': self.state_dict(),
            'd_state': self.d_state,
            'd_embed': self.d_embed,
            'd_hidden': self.d_hidden,
            'n_layers': self.trajectory_encoder.num_layers,
            'n_heads': self.trajectory_encoder.layers[0].self_attn.num_heads,
        }, path)
        print(f"💾 OutcomeEmbeddingModel saved to {path}")

    @classmethod
    def load(cls, path: str, device: torch.device = None) -> 'OutcomeEmbeddingModel':
        """Load model from checkpoint"""
        if device is None:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            
        checkpoint = torch.load(path, map_location=device)
        
        model = cls(
            d_state=checkpoint.get('d_state', 76),
            d_embed=checkpoint.get('d_embed', 64),
            d_hidden=checkpoint.get('d_hidden', 256),
            n_layers=checkpoint.get('n_layers', 3),
            n_heads=checkpoint.get('n_heads', 8)
        ).to(device)
        
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()
        
        print(f"📂 OutcomeEmbeddingModel loaded from {path}")
        return model

In [5]:
# PRE-TRAINER

class OutcomeEmbeddingPretrainer:

    def __init__(self, model: OutcomeEmbeddingModel, device: str = 'cuda'):
        self.model = model
        self.device = device
        
    def pretrain(self,
                 train_loader: torch.utils.data.DataLoader,
                 val_loader: torch.utils.data.DataLoader,
                 n_epochs: int = 30,
                 lr: float = 1e-3,
                 save_path: str = 'outcome_model.pt'):
        
        optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=lr,
            weight_decay=0.01
        )
        
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
        
        best_val_auc = 0
        
        print(f"\n Pre-training Outcome Model for {n_epochs} epochs...")
        
        for epoch in range(n_epochs):
            train_loss, train_auc, train_metrics = self._train_epoch(train_loader, optimizer)
            val_loss, val_auc, val_metrics = self._validate(val_loader)
            
            scheduler.step()
            
            print(f"\nEpoch {epoch+1}/{n_epochs}")
            print(f"  Train: Loss={train_loss:.4f} | Mort={train_metrics['mortality']:.4f} | "
                  f"Contr={train_metrics['contrastive']:.4f} | AUC={train_auc:.3f}")
            print(f"  Val:   Loss={val_loss:.4f} | Mort={val_metrics['mortality']:.4f} | "
                  f"Contr={val_metrics['contrastive']:.4f} | AUC={val_auc:.3f}")
            
            if val_auc > best_val_auc:
                best_val_auc = val_auc
                torch.save({
                    'model': self.model.state_dict(),
                    'epoch': epoch,
                    'val_auc': val_auc
                }, save_path)
                print(f"  ✅ Saved (AUC: {val_auc:.3f})")
        
        checkpoint = torch.load(save_path)
        self.model.load_state_dict(checkpoint['model'])
        
        print("\n📊 Computing 'good outcome' reference...")
        self._compute_reference_embedding(train_loader)
        
        print(f"\n✅ Pre-training complete! Best AUC: {best_val_auc:.3f}")
        
        return self.model
    
    def _train_epoch(self, loader, optimizer):
        self.model.train()
        
        total_loss = 0
        total_mortality_loss = 0
        total_contrastive_loss = 0
        
        all_preds = []
        all_labels = []
        
        for batch in tqdm(loader, desc="Training", leave=False):
            future_states = batch['future_states'].to(self.device)
            mortality = batch['mortality'].to(self.device)
            mask = batch['future_mask'].to(self.device)
            
            outcome_embed, mortality_logit = self.model(future_states, mask)
            
            # Loss 1: Mortality prediction
            loss_mortality = F.binary_cross_entropy_with_logits(
                mortality_logit.squeeze(),
                mortality.float()
            )
            
            # Loss 2: Contrastive learning
            survived_mask = (mortality == 0)
            died_mask = (mortality == 1)
            
            if survived_mask.sum() > 1 and died_mask.sum() > 1:
                survived_embeds = outcome_embed[survived_mask]
                died_embeds = outcome_embed[died_mask]
                
                # Within-group similarity (maximize)
                within_survived = torch.mm(survived_embeds, survived_embeds.T)
                within_died = torch.mm(died_embeds, died_embeds.T)
                
                # Between-group similarity (minimize)
                between = torch.mm(survived_embeds, died_embeds.T)
                
                loss_contrastive = -(
                    within_survived.mean() + within_died.mean() - 2 * between.mean()
                )
            else:
                loss_contrastive = torch.tensor(0.0, device=self.device)
            
            loss = loss_mortality + 0.1 * loss_contrastive
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            optimizer.step()
            
            total_loss += loss.item()
            total_mortality_loss += loss_mortality.item()
            total_contrastive_loss += loss_contrastive.item()
            
            with torch.no_grad():
                preds = torch.sigmoid(mortality_logit).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(mortality.cpu().numpy())
        
        auc = roc_auc_score(all_labels, all_preds)
        
        n = len(loader)
        return total_loss / n, auc, {
            'mortality': total_mortality_loss / n,
            'contrastive': total_contrastive_loss / n
        }
    
    def _validate(self, loader):
        self.model.eval()
        
        total_loss = 0
        total_mortality_loss = 0
        total_contrastive_loss = 0
        
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for batch in loader:
                future_states = batch['future_states'].to(self.device)
                mortality = batch['mortality'].to(self.device)
                mask = batch['future_mask'].to(self.device)
                
                outcome_embed, mortality_logit = self.model(future_states, mask)
                
                loss_mortality = F.binary_cross_entropy_with_logits(
                    mortality_logit.squeeze(),
                    mortality.float()
                )
                
                survived_mask = (mortality == 0)
                died_mask = (mortality == 1)
                
                if survived_mask.sum() > 1 and died_mask.sum() > 1:
                    survived_embeds = outcome_embed[survived_mask]
                    died_embeds = outcome_embed[died_mask]
                    
                    within_survived = torch.mm(survived_embeds, survived_embeds.T)
                    within_died = torch.mm(died_embeds, died_embeds.T)
                    between = torch.mm(survived_embeds, died_embeds.T)
                    
                    loss_contrastive = -(
                        within_survived.mean() + within_died.mean() - 2 * between.mean()
                    )
                else:
                    loss_contrastive = torch.tensor(0.0)
                
                loss = loss_mortality + 0.1 * loss_contrastive
                
                total_loss += loss.item()
                total_mortality_loss += loss_mortality.item()
                total_contrastive_loss += loss_contrastive.item()
                
                preds = torch.sigmoid(mortality_logit).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(mortality.cpu().numpy())
        
        auc = roc_auc_score(all_labels, all_preds)
        
        n = len(loader)
        return total_loss / n, auc, {
            'mortality': total_mortality_loss / n,
            'contrastive': total_contrastive_loss / n
        }
    
    def _compute_reference_embedding(self, loader):
        """
        Compute "good outcome" reference from survivors with stable SOFA.
        
        """
        self.model.eval()
        good_embeds = []
        
        with torch.no_grad():
            for batch in loader:
                future_states = batch['future_states'].to(self.device)
                mortality = batch['mortality'].to(self.device)
                mask = batch['future_mask'].to(self.device)
                
                if 'sofa_improved' in batch:
                    sofa_improved = batch['sofa_improved'].to(self.device)
                else:
                    sofa_improved = torch.ones_like(mortality).bool()
                
                outcome_embed, _ = self.model(future_states, mask)
                
                # Good outcomes: survived AND SOFA stable/improved
                good_mask = (mortality == 0) & sofa_improved
                
                if good_mask.sum() > 0:
                    good_embeds.append(outcome_embed[good_mask])
        
        if len(good_embeds) > 0:
            good_embeds = torch.cat(good_embeds, dim=0)
            reference = good_embeds.mean(dim=0)
            reference = F.normalize(reference, dim=-1)
            
            self.model.good_outcome_reference = reference
            
            print(f"  ✅ Reference from {len(good_embeds)} good outcomes")
        else:
            print("  ⚠️  No good outcomes found")

In [6]:
# DATALOADER UTILITIES

class OutcomeDataset(torch.utils.data.Dataset):
    """Dataset for outcome embedding pre-training"""
    def __init__(self, trajectories: List, future_window: int = 24):
        self.trajectories = trajectories
        self.future_window = future_window
        
        if len(trajectories) > 0:
            actual_dim = trajectories[0].states.shape[1]
            if actual_dim != 70:
                print(f"⚠️  State dim: {actual_dim}, expected 70")
        
    def __len__(self):
        return len(self.trajectories)
    
    def __getitem__(self, idx):
        traj = self.trajectories[idx]
        
        future_length = min(self.future_window, traj.length)
        future_states = traj.states[:future_length]
        
        sofa_improved = traj.max_sofa <= (traj.initial_sofa + 2)
        
        return {
            'future_states': torch.from_numpy(future_states).float(),
            'future_length': future_length,
            'mortality': int(traj.mortality_48h),
            'sofa_improved': sofa_improved
        }


def collate_outcomes(batch):
    """Collate with padding"""
    max_length = max(item['future_length'] for item in batch)
    
    future_states_padded = []
    future_masks = []
    
    for item in batch:
        T = item['future_length']
        D_state = item['future_states'].shape[1]
        
        mask = torch.zeros(max_length)
        mask[:T] = 1
        
        if T < max_length:
            padding = torch.zeros(max_length - T, D_state)
            states = torch.cat([item['future_states'], padding], dim=0)
        else:
            states = item['future_states']
        
        future_states_padded.append(states)
        future_masks.append(mask)
    
    return {
        'future_states': torch.stack(future_states_padded),
        'future_mask': torch.stack(future_masks),
        'mortality': torch.tensor([item['mortality'] for item in batch]),
        'sofa_improved': torch.tensor([item['sofa_improved'] for item in batch])
    }


def create_outcome_dataloaders(train_trajectories: List,
                                 val_trajectories: List,
                                 batch_size: int = 32,
                                 future_window: int = 24):
    """Create dataloaders"""
    train_dataset = OutcomeDataset(train_trajectories, future_window)
    val_dataset = OutcomeDataset(val_trajectories, future_window)
    
    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_outcomes,
        num_workers=0
    )
    
    val_loader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_outcomes,
        num_workers=0
    )
    
    return train_loader, val_loader

In [7]:
if __name__ == "__main__":
    # Step 1: Extract cohort
    extractor = CohortExtractor()
    cohort_df = extractor.extract_cohort()
    
    # Step 2: Initialize components
    feature_extractor = FeatureExtractor(FeatureConfig())
    
    action_discretizer = ActionDiscretizer(
        bins_config=ActionBins(), 
        data_dir='../../../../../data/' 
    )
    
    feature_imputer = fit_imputer_on_cohort(cohort_df, feature_extractor)
    
    # Step 3: Build trajectories
    traj_builder = TrajectoryBuilder(
        feature_extractor,
        action_discretizer,
        feature_imputer,
        min_length=24,
        max_length=168
    )
    trajectories = traj_builder.build_dataset(cohort_df)
    
    # Step 4: Split
    splitter = DatasetSplitter()
    splits = splitter.split(trajectories)
    
    # Step 5: Normalize
    normalizer = FeatureNormalizer()
    normalizer.fit(splits['train'])
    normalized_train = normalizer.transform(splits['train'])
    normalized_val = normalizer.transform(splits['val'])
    
    train_loader, val_loader = create_outcome_dataloaders(
        normalized_train,
        normalized_val,
        batch_size=32,
        future_window=24
    )
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')   

No DB config provided. Using CSV mode.


Angus sepsis cases identified: 76 hospital admissions
✅ Extracted 51 Angus sepsis trajectories (adult, LOS 24-168h)


INFO:utils.data_pipeline:ActionDiscretizer initialized with data for 75 icustays.



🔧 Fitting FeatureImputer on raw cohort data...


  Learned population statistics from 3082 training samples
✅ Imputer fitted successfully on 3,082 hourly records
🔧 Building trajectories...


Processing patients: 100%|██████████| 51/51 [02:53<00:00,  3.41s/it]


🔍 Trajectory Debug Info:

ICU Stay 206504:
  State dim: 38
  SOFA (initial/max): 5.0/5.0
  Mortality: False
  Non-NaN features: 1520 / 1520

ICU Stay 264446:
  State dim: 38
  SOFA (initial/max): 5.0/5.0
  Mortality: False
  Non-NaN features: 2432 / 2432

ICU Stay 228977:
  State dim: 38
  SOFA (initial/max): 5.0/5.0
  Mortality: True
  Non-NaN features: 1216 / 1216
✅ Built 51 valid trajectories
⚠️  Failed 0 quality checks

📊 Dataset Split Statistics:
  TRAIN:   35 trajectories | Mortality: 17.1% | Avg Length: 56.6h | Avg SOFA: 5.0
  VAL  :    7 trajectories | Mortality: 14.3% | Avg Length: 65.6h | Avg SOFA: 5.0
  TEST :    9 trajectories | Mortality: 0.0% | Avg Length: 71.3h | Avg SOFA: 5.0

📐 Feature Normalization Statistics:
  State dimension: 38
  Mean range: [0.00, 73.40]
  Std range: [0.00, 28.33]
⚠️  State dim: 38, expected 70
⚠️  State dim: 38, expected 70
